In [0]:
%sql
CREATE OR REPLACE TABLE Employees (
    emp_id INT,
    name STRING,
    salary INT,
    bonus INT,
    manager_id INT
);

INSERT INTO Employees
SELECT * FROM (VALUES
(1, 'Amit', 50000, NULL, 101),
(2, 'John', NULL, 5000, 102),
(3, 'Sara', 60000, NULL, NULL),
(4, 'David', NULL, NULL, 103),
(5, 'Priya', 45000, 3000, 101),
(6, 'Kiran', NULL, NULL, NULL),
(7, 'Ravi', 70000, 7000, 102),
(8, 'Neha', NULL, 2000, NULL)
) AS v(emp_id, name, salary, bonus, manager_id);

CREATE OR REPLACE TABLE Orders (
    order_id INT,
    customer_name STRING,
    amount INT,
    discount INT,
    coupon_code STRING
);

INSERT INTO Orders
SELECT * FROM (VALUES
(101, 'Amit', 1000, NULL, 'DISC10'),
(102, 'John', NULL, 50, NULL),
(103, 'Sara', 2000, NULL, 'DISC20'),
(104, 'David', NULL, NULL, NULL),
(105, 'Priya', 1500, 100, NULL),
(106, 'Kiran', NULL, NULL, 'DISC5'),
(107, 'Ravi', 3000, NULL, NULL),
(108, 'Neha', NULL, 200, 'DISC15')
) AS v(order_id, customer_name, amount, discount, coupon_code);

CREATE OR REPLACE TABLE Products (
    product_id INT,
    product_name STRING,
    price INT,
    category STRING,
    stock INT
);

INSERT INTO Products
SELECT * FROM (VALUES
(1, 'Laptop', 50000, 'Electronics', 10),
(2, 'Phone', NULL, 'Electronics', NULL),
(3, 'Tablet', 30000, NULL, 5),
(4, 'Headphones', NULL, NULL, NULL),
(5, 'Monitor', 20000, 'Electronics', 0),
(6, 'Keyboard', NULL, 'Accessories', 15),
(7, 'Mouse', 500, NULL, NULL),
(8, 'Printer', NULL, 'Electronics', 3)
) AS v(product_id, product_name, price, category, stock);

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
--Employees whose salary is NULL
SELECT * FROM Employees
WHERE salary IS NULL;

emp_id,name,salary,bonus,manager_id
2,John,null,5000,102
4,David,null,null,103
6,Kiran,null,null,null
8,Neha,null,2000,null


In [0]:
%sql
--Orders where discount is NOT NULL
SELECT * FROM Orders
WHERE discount IS NOT NULL;


order_id,customer_name,amount,discount,coupon_code
102,John,null,50,null
105,Priya,1500,100,null
108,Neha,null,200,DISC15


In [0]:
%sql
--Products where category is NULL
SELECT * FROM Products
WHERE category IS NULL;


product_id,product_name,price,category,stock
3,Tablet,30000,null,5
4,Headphones,null,null,null
7,Mouse,500,null,null


In [0]:
%sql
-- Count employees with NULL manager_id
SELECT COUNT(*) 
FROM Employees
WHERE manager_id IS NULL;

COUNT(*)
3


In [0]:
%sql
-- Replace NULL salary with 0
UPDATE Employees
SET salary = 0
WHERE salary IS NULL;


num_affected_rows
4


In [0]:
%sql

-- Replace NULL bonus with 1000
UPDATE Employees
SET bonus = 1000
WHERE bonus IS NULL;



num_affected_rows
4


In [0]:
%sql
-- Show order amount, if NULL replace with 500
UPDATE Orders
SET amount = 500
WHERE amount IS NULL;


num_affected_rows
4


In [0]:
%sql
-- Replace NULL stock with 0
UPDATE Products
SET stock = 0
WHERE stock IS NULL;

num_affected_rows
3


In [0]:
%sql
-- Show employee earnings using: salary, if NULL use bonus
SELECT emp_id, name, COALESCE(salary, bonus) AS earnings
FROM Employees;


emp_id,name,earnings
4,David,0
6,Kiran,0
1,Amit,50000
3,Sara,60000
2,John,0
8,Neha,0
5,Priya,45000
7,Ravi,70000


In [0]:
%sql

-- Show first available value: salary → bonus → 0
SELECT emp_id, name, COALESCE(salary, bonus, 0) AS income
FROM Employees;


emp_id,name,income
4,David,0
6,Kiran,0
1,Amit,50000
3,Sara,60000
2,John,0
8,Neha,0
5,Priya,45000
7,Ravi,70000


In [0]:
%sql

-- Show product price: price → 1000 (default)
SELECT product_id, product_name, COALESCE(price, 1000) AS price
FROM Products;



product_id,product_name,price
2,Phone,1000
4,Headphones,1000
7,Mouse,500
1,Laptop,50000
3,Tablet,30000
5,Monitor,20000
6,Keyboard,1000
8,Printer,1000


In [0]:
%sql
-- Get customer payment: amount → discount → 0
SELECT order_id, customer_name, COALESCE(amount, discount, 0) AS payment
FROM Orders;

order_id,customer_name,payment
102,John,500
104,David,500
106,Kiran,500
108,Neha,500
101,Amit,1000
103,Sara,2000
105,Priya,1500
107,Ravi,3000


In [0]:
%sql
-- Convert salary to NULL if salary = 0
SELECT emp_id, name, NULLIF(salary, 0) AS salary
FROM Employees;


emp_id,name,salary
4,David,null
6,Kiran,null
1,Amit,50000
3,Sara,60000
2,John,null
8,Neha,null
5,Priya,45000
7,Ravi,70000


In [0]:
%sql

-- Convert discount to NULL if discount = 0
SELECT order_id, NULLIF(discount, 0) AS discount
FROM Orders;


order_id,discount
102,50
104,null
106,null
108,200
101,null
103,null
105,100
107,null


In [0]:
%sql

-- Use NULLIF to avoid divide by zero
SELECT amount / NULLIF(discount, 0) AS result
FROM Orders;


result
10.0
null
null
2.5
null
null
15.0
null


In [0]:
%sql

-- Replace coupon_code with NULL if it is 'DISC10'
SELECT order_id, NULLIF(coupon_code, 'DISC10') AS coupon_code
FROM Orders;

order_id,coupon_code
102,null
104,null
106,DISC5
108,DISC15
101,null
103,DISC20
105,null
107,null


In [0]:
%sql
-- Calculate total earnings: salary + bonus (handle NULL properly)
SELECT emp_id, name,
COALESCE(salary, 0) + COALESCE(bonus, 0) AS total_earnings
FROM Employees;


emp_id,name,total_earnings
4,David,1000
6,Kiran,1000
1,Amit,51000
3,Sara,61000
2,John,5000
8,Neha,2000
5,Priya,48000
7,Ravi,77000


In [0]:
%sql

-- Show employees where both salary AND bonus are NULL
SELECT *
FROM Employees
WHERE salary IS NULL AND bonus IS NULL;


emp_id,name,salary,bonus,manager_id


In [0]:
%sql
-- Show products where price is NULL but category is NOT NULL
SELECT *
FROM Products
WHERE price IS NULL AND category IS NOT NULL;


product_id,product_name,price,category,stock
2,Phone,null,Electronics,0
6,Keyboard,null,Accessories,15
8,Printer,null,Electronics,3


In [0]:
%sql
-- Show orders where both amount and discount are NULL
SELECT *
FROM Orders
WHERE amount IS NULL AND discount IS NULL;

order_id,customer_name,amount,discount,coupon_code


In [0]:
%sql
-- Show employee income: COALESCE(salary, bonus, 1000)
SELECT emp_id, name,
COALESCE(salary, bonus, 1000) AS income
FROM Employees;


emp_id,name,income
4,David,0
6,Kiran,0
1,Amit,50000
3,Sara,60000
2,John,0
8,Neha,0
5,Priya,45000
7,Ravi,70000


In [0]:
%sql

-- Replace empty discount with NULL using NULLIF
SELECT order_id, NULLIF(discount, 0) AS discount
FROM Orders;


order_id,discount
102,50
104,null
106,null
108,200
101,null
103,null
105,100
107,null


In [0]:
%sql
-- Show final payable amount: amount - discount (handle NULL)
SELECT order_id,
COALESCE(amount, 0) - COALESCE(discount, 0) AS payable_amount
FROM Orders;


order_id,payable_amount
102,450
104,500
106,500
108,300
101,1000
103,2000
105,1400
107,3000


In [0]:
%sql
-- Find employees where salary is NULL but manager exists
SELECT *
FROM Employees
WHERE salary IS NULL AND manager_id IS NOT NULL;

emp_id,name,salary,bonus,manager_id
